# 1. Import Libraries

In [47]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 150)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

# 2. Define Paths

In [48]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DATA_ARTIFACT_DIR = PROJECT_ROOT / "artifacts" / "data"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

DATA_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_GRADE_PATH = PROCESSED_DIR / "train_scored_with_grades.csv"
VALID_GRADE_PATH = PROCESSED_DIR / "valid_scored_with_grades.csv"
TEST_GRADE_PATH = PROCESSED_DIR / "test_scored_with_grades.csv"

print("Project root:", PROJECT_ROOT)
print("Train with grades exists:", TRAIN_GRADE_PATH.exists())
print("Valid with grades exists:", VALID_GRADE_PATH.exists())
print("Test with grades exists :", TEST_GRADE_PATH.exists())

Project root: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator
Train with grades exists: True
Valid with grades exists: True
Test with grades exists : True


# 3. Load Data

In [49]:
train_df = pd.read_csv(TRAIN_GRADE_PATH, low_memory=False, parse_dates=["issue_date"])
valid_df = pd.read_csv(VALID_GRADE_PATH, low_memory=False, parse_dates=["issue_date"])
test_df = pd.read_csv(TEST_GRADE_PATH, low_memory=False, parse_dates=["issue_date"])

print("Train:", train_df.shape)
print("Valid:", valid_df.shape)
print("Test :", test_df.shape)

required_cols = [
    "loan_amnt",
    "term_months",
    "int_rate",
    "default_flag",
    "calibrated_pd",
    "internal_grade"
]

for col in required_cols:
    if col not in test_df.columns:
        raise ValueError(f"Missing required column: {col}")

display(test_df.head())

Train: (277221, 33)
Valid: (59404, 33)
Test : (59405, 33)


,default_flag,loan_amnt,term_months,annual_inc_log,dti_capped,emp_length_num,emp_length_missing,open_acc,pub_rec,revol_bal_log,revol_util_capped,revol_util_missing,total_acc,mort_acc,mort_acc_missing,pub_rec_bankruptcies,pub_rec_bankruptcies_missing,credit_history_length_capped,home_ownership,verification_status,purpose,initial_list_status,application_type,loan_status,grade,sub_grade,int_rate,installment,issue_date,pd_raw,pd_sigmoid,calibrated_pd,internal_grade
0,0,"5,000.0000",36.0000,10.0859,28.5500,4.0000,0,10.0000,1.0000,8.4065,49.0000,0,34.0000,4.0000,0,1.0000,0,24.5886,RENT,Not Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,C,C1,12.3900,167.0100,2015-02-01,0.2569,0.2372,0.2569,B
1,0,"25,000.0000",36.0000,11.3504,12.6500,7.0000,0,13.0000,0.0000,9.6637,43.5000,0,25.0000,2.0000,0,0.0000,0,10.9158,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,D,D4,17.5700,898.4300,2014-08-01,0.1147,0.1176,0.1147,A
2,0,"11,000.0000",36.0000,11.2898,17.6000,10.0000,0,19.0000,0.0000,10.2295,66.8000,0,44.0000,3.0000,0,0.0000,0,12.0000,MORTGAGE,Not Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,C,C1,14.3000,377.5600,2013-10-01,0.1286,0.1265,0.1286,A
3,1,"23,325.0000",60.0000,11.3386,31.8600,10.0000,0,16.0000,0.0000,6.8480,4.8000,0,26.0000,2.0000,0,0.0000,0,17.4155,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Charged Off,B,B4,13.1100,532.0300,2013-04-01,0.3215,0.3136,0.3215,CCC
4,0,"10,000.0000",36.0000,11.2898,19.0800,1.0000,0,12.0000,0.0000,7.5507,8.0000,0,55.0000,3.0000,0,0.0000,0,12.0767,MORTGAGE,Not Verified,house,f,INDIVIDUAL,Fully Paid,A,A1,6.0300,304.3600,2013-03-01,0.0663,0.0908,0.0663,AAA


# 4. Define Business Assumptions

In [50]:
ASSUMPTIONS = {
    "funding_cost_rate": 0.04,          # annual funding cost
    "operating_cost_rate": 0.02,        # annual operating cost
    "target_margin_rate": 0.05,         # target margin for required rate
    "pricing_tolerance": 0.01,          # 1 percentage point tolerance
    "max_allowed_rate": 0.30,
    "min_expected_profit": 0.0,

    
    "capital_cost_rate": 0.08,          # annual cost of capital allocated to risky exposure
    "collection_cost_rate": 0.10,       # collection cost applied to expected loss
    "tail_risk_multiplier": 0.35        # extra penalty for high-PD loans
}

# 5. Define LGD Assumption by Internal Grade

In [51]:
# Higher-risk grades are assigned higher LGD assumptions.
# This is a simulation assumption, not an estimated LGD model.

LGD_BY_GRADE = {
    "AAA": 0.30,
    "AA": 0.32,
    "A": 0.35,
    "BBB": 0.40,
    "BB": 0.45,
    "B": 0.50,
    "CCC": 0.55,
    "D": 0.60
}

lgd_table = pd.DataFrame({
    "internal_grade": list(LGD_BY_GRADE.keys()),
    "lgd_assumption": list(LGD_BY_GRADE.values())
})

display(lgd_table)

,internal_grade,lgd_assumption
0,AAA,0.3000
1,AA,0.3200
2,A,0.3500
3,BBB,0.4000
4,BB,0.4500
5,B,0.5000
6,CCC,0.5500
7,D,0.6000


# 6. Pricing and Profitability Functions

In [52]:
def add_pricing_profitability_metrics(
    df,
    lgd_by_grade,
    funding_cost_rate=0.04,
    operating_cost_rate=0.02,
    target_margin_rate=0.05,
    pricing_tolerance=0.01,
    max_allowed_rate=0.30,
    min_expected_profit=0.0,
    capital_cost_rate=0.08,
    collection_cost_rate=0.10,
    tail_risk_multiplier=0.35,
    pd_col="calibrated_pd",
    grade_col="internal_grade",
    ead_col="loan_amnt",
    actual_rate_col="int_rate",
    term_col="term_months"
):
    """
    Add risk-based pricing, accounting-style expected profit, and risk-adjusted economic profit.

    Modeling convention:
    - PD is treated as lifetime default probability.
    - Expected loss = PD × LGD × EAD.
    - Expected loss rate is annualized for required annual rate comparison.
    - Economic profit adds capital charge, collection cost, and tail-risk penalty.
    """
    out = df.copy()

    out["actual_rate"] = out[actual_rate_col] / 100.0
    out["ead"] = out[ead_col]
    out["term_years"] = out[term_col] / 12.0

    out["lgd"] = out[grade_col].map(lgd_by_grade)

    if out["lgd"].isna().any():
        missing_grades = out.loc[out["lgd"].isna(), grade_col].unique()
        raise ValueError(f"Missing LGD mapping for grades: {missing_grades}")

    # Expected loss
    out["expected_loss"] = out[pd_col] * out["lgd"] * out["ead"]
    out["lifetime_expected_loss_rate"] = out[pd_col] * out["lgd"]
    out["annualized_expected_loss_rate"] = out["lifetime_expected_loss_rate"] / out["term_years"]

    # Required annual risk-based interest rate
    out["required_rate"] = (
        funding_cost_rate
        + operating_cost_rate
        + out["annualized_expected_loss_rate"]
        + target_margin_rate
    )

    out["required_rate_capped"] = out["required_rate"].clip(upper=max_allowed_rate)
    out["pricing_gap"] = out["actual_rate"] - out["required_rate"]

    out["pricing_status"] = np.select(
        [
            out["pricing_gap"] < -pricing_tolerance,
            out["pricing_gap"].between(-pricing_tolerance, pricing_tolerance, inclusive="both"),
            out["pricing_gap"] > pricing_tolerance
        ],
        [
            "Underpriced",
            "Fairly Priced",
            "Overpriced"
        ],
        default="Unknown"
    )

    # Accounting-style profit proxy
    out["interest_income"] = out["ead"] * out["actual_rate"] * out["term_years"]
    out["funding_cost"] = out["ead"] * funding_cost_rate * out["term_years"]
    out["operating_cost"] = out["ead"] * operating_cost_rate * out["term_years"]

    out["expected_profit"] = (
        out["interest_income"]
        - out["funding_cost"]
        - out["operating_cost"]
        - out["expected_loss"]
    )

    out["risk_adjusted_return"] = out["expected_profit"] / out["ead"]

    # Repriced profit
    out["repriced_interest_income"] = out["ead"] * out["required_rate_capped"] * out["term_years"]

    out["repriced_expected_profit"] = (
        out["repriced_interest_income"]
        - out["funding_cost"]
        - out["operating_cost"]
        - out["expected_loss"]
    )

    out["repriced_risk_adjusted_return"] = out["repriced_expected_profit"] / out["ead"]

    # Economic profit adjustment

    # Capital requirement proxy:
    # higher PD and LGD require more economic capital.
    out["capital_requirement"] = out["ead"] * out[pd_col] * out["lgd"]

    out["capital_charge"] = (
        out["capital_requirement"]
        * capital_cost_rate
        * out["term_years"]
    )

    # Collection cost proxy:
    # expected collection burden scales with expected loss.
    out["collection_cost"] = out["expected_loss"] * collection_cost_rate

    # Tail-risk penalty:
    # penalizes very high-PD loans beyond linear expected loss.
    out["tail_risk_penalty"] = (
        out["ead"]
        * np.square(out[pd_col])
        * out["lgd"]
        * tail_risk_multiplier
    )

    out["economic_profit"] = (
        out["expected_profit"]
        - out["capital_charge"]
        - out["collection_cost"]
        - out["tail_risk_penalty"]
    )

    out["economic_return"] = out["economic_profit"] / out["ead"]

    out["repriced_economic_profit"] = (
        out["repriced_expected_profit"]
        - out["capital_charge"]
        - out["collection_cost"]
        - out["tail_risk_penalty"]
    )

    out["repriced_economic_return"] = out["repriced_economic_profit"] / out["ead"]

    # Decision logic should use economic profit, not simple expected profit
    approve_current = (
        (out["economic_profit"] >= min_expected_profit)
        & (out["required_rate"] <= max_allowed_rate)
    )

    approve_if_repriced = (
        (out["economic_profit"] < min_expected_profit)
        & (out["repriced_economic_profit"] >= min_expected_profit)
        & (out["required_rate"] <= max_allowed_rate)
    )

    manual_review = (
        (out["required_rate"] > max_allowed_rate)
        & (out[pd_col] < 0.35)
    )

    reject = ~(approve_current | approve_if_repriced | manual_review)

    out["pricing_decision"] = np.select(
        [
            approve_current,
            approve_if_repriced,
            manual_review,
            reject
        ],
        [
            "Approve at Current Rate",
            "Approve if Repriced",
            "Manual Review",
            "Reject"
        ],
        default="Unknown"
    )

    return out

# 7. Pricing Engine

In [53]:
# Apply pricing engine to train/valid/test

train_pricing = add_pricing_profitability_metrics(
    train_df,
    LGD_BY_GRADE,
    **ASSUMPTIONS
)

valid_pricing = add_pricing_profitability_metrics(
    valid_df,
    LGD_BY_GRADE,
    **ASSUMPTIONS
)

test_pricing = add_pricing_profitability_metrics(
    test_df,
    LGD_BY_GRADE,
    **ASSUMPTIONS
)

print("Train pricing:", train_pricing.shape)
print("Valid pricing:", valid_pricing.shape)
print("Test pricing :", test_pricing.shape)

display(test_pricing.head())

Train pricing: (277221, 61)
Valid pricing: (59404, 61)
Test pricing : (59405, 61)


,default_flag,loan_amnt,term_months,annual_inc_log,dti_capped,emp_length_num,emp_length_missing,open_acc,pub_rec,revol_bal_log,revol_util_capped,revol_util_missing,total_acc,mort_acc,mort_acc_missing,pub_rec_bankruptcies,pub_rec_bankruptcies_missing,credit_history_length_capped,home_ownership,verification_status,purpose,initial_list_status,application_type,loan_status,grade,sub_grade,int_rate,installment,issue_date,pd_raw,pd_sigmoid,calibrated_pd,internal_grade,actual_rate,ead,term_years,lgd,expected_loss,lifetime_expected_loss_rate,annualized_expected_loss_rate,required_rate,required_rate_capped,pricing_gap,pricing_status,interest_income,funding_cost,operating_cost,expected_profit,risk_adjusted_return,repriced_interest_income,repriced_expected_profit,repriced_risk_adjusted_return,capital_requirement,capital_charge,collection_cost,tail_risk_penalty,economic_profit,economic_return,repriced_economic_profit,repriced_economic_return,pricing_decision
0,0,"5,000.0000",36.0000,10.0859,28.5500,4.0000,0,10.0000,1.0000,8.4065,49.0000,0,34.0000,4.0000,0,1.0000,0,24.5886,RENT,Not Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,C,C1,12.3900,167.0100,2015-02-01,0.2569,0.2372,0.2569,B,0.1239,"5,000.0000",3.0000,0.5000,642.2466,0.1284,0.0428,0.1528,0.1528,-0.0289,Underpriced,"1,858.5000",600.0000,300.0000,316.2534,0.0633,"2,292.2466",750.0000,0.1500,642.2466,154.1392,64.2247,57.7473,40.1423,0.0080,473.8889,0.0948,Approve at Current Rate
1,0,"25,000.0000",36.0000,11.3504,12.6500,7.0000,0,13.0000,0.0000,9.6637,43.5000,0,25.0000,2.0000,0,0.0000,0,10.9158,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Fully Paid,D,D4,17.5700,898.4300,2014-08-01,0.1147,0.1176,0.1147,A,0.1757,"25,000.0000",3.0000,0.3500,"1,004.0153",0.0402,0.0134,0.1234,0.1234,0.0523,Overpriced,"13,177.5000","3,000.0000","1,500.0000","7,673.4847",0.3069,"9,254.0153","3,750.0000",0.1500,"1,004.0153",240.9637,100.4015,40.3219,"7,291.7977",0.2917,"3,368.3130",0.1347,Approve at Current Rate
2,0,"11,000.0000",36.0000,11.2898,17.6000,10.0000,0,19.0000,0.0000,10.2295,66.8000,0,44.0000,3.0000,0,0.0000,0,12.0000,MORTGAGE,Not Verified,debt_consolidation,w,INDIVIDUAL,Fully Paid,C,C1,14.3000,377.5600,2013-10-01,0.1286,0.1265,0.1286,A,0.1430,"11,000.0000",3.0000,0.3500,495.2668,0.0450,0.0150,0.1250,0.1250,0.0180,Overpriced,"4,719.0000","1,320.0000",660.0000,"2,243.7332",0.2040,"4,125.2668","1,650.0000",0.1500,495.2668,118.8640,49.5267,22.2990,"2,053.0435",0.1866,"1,459.3103",0.1327,Approve at Current Rate
3,1,"23,325.0000",60.0000,11.3386,31.8600,10.0000,0,16.0000,0.0000,6.8480,4.8000,0,26.0000,2.0000,0,0.0000,0,17.4155,MORTGAGE,Verified,debt_consolidation,f,INDIVIDUAL,Charged Off,B,B4,13.1100,532.0300,2013-04-01,0.3215,0.3136,0.3215,CCC,0.1311,"23,325.0000",5.0000,0.5500,"4,123.8173",0.1768,0.0354,0.1454,0.1454,-0.0143,Underpriced,"15,289.5375","4,665.0000","2,332.5000","4,168.2202",0.1787,"16,952.5673","5,831.2500",0.2500,"4,123.8173","1,649.5269",412.3817,463.9621,"1,642.3494",0.0704,"3,305.3792",0.1417,Approve at Current Rate
4,0,"10,000.0000",36.0000,11.2898,19.0800,1.0000,0,12.0000,0.0000,7.5507,8.0000,0,55.0000,3.0000,0,0.0000,0,12.0767,MORTGAGE,Not Verified,house,f,INDIVIDUAL,Fully Paid,A,A1,6.0300,304.3600,2013-03-01,0.0663,0.0908,0.0663,AAA,0.0603,"10,000.0000",3.0000,0.3000,198.9846,0.0199,0.0066,0.1166,0.1166,-0.0563,Underpriced,"1,809.0000","1,200.0000",600.0000,-189.9846,-0.0190,"3,498.9846","1,500.0000",0.1500,198.9846,47.7563,19.8985,4.6194,-262.2588,-0.0262,"1,427.7258",0.1428,Approve if Repriced


In [54]:
def portfolio_kpi_summary(df, name="portfolio"):
    total_loans = len(df)
    total_ead = df["ead"].sum()

    summary = {
        "portfolio": name,
        "n_loans": total_loans,
        "total_ead": total_ead,
        "avg_pd": df["calibrated_pd"].mean(),
        "avg_lgd": df["lgd"].mean(),
        "avg_actual_rate": df["actual_rate"].mean(),
        "avg_required_rate": df["required_rate"].mean(),
        "avg_lifetime_el_rate": df["lifetime_expected_loss_rate"].mean(),
        "avg_annualized_el_rate": df["annualized_expected_loss_rate"].mean(),

        "total_interest_income": df["interest_income"].sum(),
        "total_expected_loss": df["expected_loss"].sum(),
        "total_funding_cost": df["funding_cost"].sum(),
        "total_operating_cost": df["operating_cost"].sum(),
        "total_expected_profit": df["expected_profit"].sum(),
        "portfolio_rar": df["expected_profit"].sum() / total_ead if total_ead > 0 else np.nan,

        "total_capital_charge": df["capital_charge"].sum(),
        "total_collection_cost": df["collection_cost"].sum(),
        "total_tail_risk_penalty": df["tail_risk_penalty"].sum(),
        "total_economic_profit": df["economic_profit"].sum(),
        "portfolio_economic_return": df["economic_profit"].sum() / total_ead if total_ead > 0 else np.nan,

        "actual_default_rate": df["default_flag"].mean(),
        "underpriced_share": (df["pricing_status"] == "Underpriced").mean(),
        "fairly_priced_share": (df["pricing_status"] == "Fairly Priced").mean(),
        "overpriced_share": (df["pricing_status"] == "Overpriced").mean()
    }

    return pd.DataFrame([summary])

In [55]:
# Pricing Status and Decision Distribution

print("Pricing status distribution:")
pricing_status_dist = (
    test_pricing["pricing_status"]
    .value_counts()
    .to_frame("count")
    .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
)

display(pricing_status_dist)

print("Pricing decision distribution:")
pricing_decision_dist = (
    test_pricing["pricing_decision"]
    .value_counts()
    .to_frame("count")
    .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
)

display(pricing_decision_dist)

Pricing status distribution:


,count,pct
pricing_status,,
Underpriced,24205,40.7457
Overpriced,23000,38.7173
Fairly Priced,12200,20.5370


Pricing decision distribution:


,count,pct
pricing_decision,,
Approve at Current Rate,50119,84.3683
Approve if Repriced,9146,15.3960
Reject,140,0.2357


In [56]:
def grade_profitability_summary(df):
    summary = (
        df
        .groupby("internal_grade")
        .agg(
            n_loans=("default_flag", "size"),
            total_ead=("ead", "sum"),
            avg_pd=("calibrated_pd", "mean"),
            actual_default_rate=("default_flag", "mean"),
            avg_lgd=("lgd", "mean"),
            avg_actual_rate=("actual_rate", "mean"),
            avg_lifetime_el_rate=("lifetime_expected_loss_rate", "mean"),
            avg_annualized_el_rate=("annualized_expected_loss_rate", "mean"),
            avg_required_rate=("required_rate", "mean"),
            avg_pricing_gap=("pricing_gap", "mean"),

            total_expected_loss=("expected_loss", "sum"),
            total_interest_income=("interest_income", "sum"),
            total_expected_profit=("expected_profit", "sum"),
            avg_rar=("risk_adjusted_return", "mean"),

            total_capital_charge=("capital_charge", "sum"),
            total_collection_cost=("collection_cost", "sum"),
            total_tail_risk_penalty=("tail_risk_penalty", "sum"),
            total_economic_profit=("economic_profit", "sum"),
            avg_economic_return=("economic_return", "mean"),

            total_repriced_expected_profit=("repriced_expected_profit", "sum"),
            avg_repriced_rar=("repriced_risk_adjusted_return", "mean"),
            total_repriced_economic_profit=("repriced_economic_profit", "sum"),
            avg_repriced_economic_return=("repriced_economic_return", "mean"),

            underpriced_share=("pricing_status", lambda x: (x == "Underpriced").mean()),
            approve_current_share=("pricing_decision", lambda x: (x == "Approve at Current Rate").mean()),
            reprice_share=("pricing_decision", lambda x: (x == "Approve if Repriced").mean()),
            manual_review_share=("pricing_decision", lambda x: (x == "Manual Review").mean()),
            reject_share=("pricing_decision", lambda x: (x == "Reject").mean())
        )
        .reset_index()
    )

    grade_order = {
        "AAA": 0,
        "AA": 1,
        "A": 2,
        "BBB": 3,
        "BB": 4,
        "B": 5,
        "CCC": 6,
        "D": 7
    }

    summary["grade_order"] = summary["internal_grade"].map(grade_order)
    summary = summary.sort_values("grade_order").drop(columns="grade_order")

    return summary


test_grade_profitability = grade_profitability_summary(test_pricing)

display(test_grade_profitability)

,internal_grade,n_loans,total_ead,avg_pd,actual_default_rate,avg_lgd,avg_actual_rate,avg_lifetime_el_rate,avg_annualized_el_rate,avg_required_rate,avg_pricing_gap,total_expected_loss,total_interest_income,total_expected_profit,avg_rar,total_capital_charge,total_collection_cost,total_tail_risk_penalty,total_economic_profit,avg_economic_return,total_repriced_expected_profit,avg_repriced_rar,total_repriced_economic_profit,avg_repriced_economic_return,underpriced_share,approve_current_share,reprice_share,manual_review_share,reject_share
2,AAA,3415,"41,277,700.0000",0.0541,0.0419,0.3000,0.0871,0.0162,0.0054,0.1154,-0.0283,"673,390.8858","10,658,971.3150","2,540,546.4292",0.0654,"161,955.5626","67,339.0886","13,236.2182","2,298,015.5597",0.0595,"6,204,195.0000",0.1504,"5,961,664.1305",0.1445,0.7587,0.7016,0.2984,0.0000,0.0000
1,AA,8488,"108,652,875.0000",0.0871,0.0778,0.3200,0.1070,0.0279,0.0092,0.1192,-0.0122,"3,031,415.9578","34,865,640.5675","11,983,910.1097",0.1151,"739,004.3883","303,141.5958","93,770.8746","10,847,993.2511",0.1047,"16,541,928.7500",0.1518,"15,406,011.8913",0.1414,0.5422,0.8446,0.1554,0.0000,0.0000
0,A,11518,"150,649,300.0000",0.1239,0.1235,0.3500,0.1223,0.0434,0.0142,0.1242,-0.0018,"6,537,967.2982","57,641,427.6775","22,791,989.3793",0.1504,"1,640,124.3987","653,796.7298","286,308.7553","20,211,759.4955",0.1334,"23,592,892.5000",0.1548,"21,012,662.6161",0.1378,0.4239,0.8703,0.1297,0.0000,0.0000
5,BBB,11905,"158,117,750.0000",0.1687,0.1661,0.4000,0.1338,0.0675,0.0214,0.1314,0.0024,"10,666,512.4991","72,490,999.9800","29,903,584.4809",0.1748,"2,878,039.7338","1,066,651.2499","634,260.9764","25,324,632.5209",0.1465,"26,600,752.5000",0.1623,"22,021,800.5400",0.1340,0.3787,0.8744,0.1256,0.0000,0.0000
4,BB,7727,"107,406,275.0000",0.2158,0.2170,0.4500,0.1455,0.0971,0.0291,0.1391,0.0063,"10,439,370.0096","60,180,022.1375","25,707,692.6279",0.2074,"3,118,753.8392","1,043,937.0010","792,010.9425","20,752,990.8452",0.1632,"20,027,466.2500",0.1748,"15,072,764.4673",0.1305,0.3484,0.8764,0.1236,0.0000,0.0000
3,B,6987,"106,093,775.0000",0.2690,0.2779,0.5000,0.1570,0.1345,0.0374,0.1474,0.0097,"14,308,314.4371","71,478,036.1450","30,962,017.2079",0.2456,"4,723,075.5291","1,430,831.4437","1,356,754.5130","23,451,355.7222",0.1782,"21,839,753.7500",0.1912,"14,329,092.2643",0.1238,0.3446,0.8496,0.1504,0.0000,0.0000
6,CCC,5333,"92,891,825.0000",0.3445,0.3531,0.5500,0.1721,0.1895,0.0462,0.1562,0.0158,"17,648,987.5629","75,919,368.3700","32,778,140.3071",0.3053,"6,469,736.8761","1,764,898.7563","2,146,165.0522","22,397,339.6225",0.1976,"21,243,533.7500",0.2164,"10,862,733.0654",0.1087,0.2929,0.8095,0.1905,0.0000,0.0000
7,D,4032,"73,525,800.0000",0.4718,0.4826,0.6000,0.1907,0.2831,0.0607,0.1707,0.0200,"20,820,035.7875","69,818,734.3450","27,501,136.5575",0.3465,"8,127,459.6865","2,082,003.5788","3,496,103.0027","13,795,570.2895",0.1624,"17,914,635.0000",0.2386,"4,209,068.7320",0.0544,0.2378,0.7676,0.1977,0.0000,0.0347


In [57]:
rate_by_grade = test_grade_profitability[
    [
        "internal_grade",
        "n_loans",
        "avg_pd",
        "actual_default_rate",
        "avg_actual_rate",
        "avg_annualized_el_rate",
        "avg_required_rate",
        "avg_pricing_gap",
        "underpriced_share",
        "total_expected_profit",
        "avg_rar",
        "total_economic_profit",
        "avg_economic_return"
    ]
].copy()

display(rate_by_grade)

,internal_grade,n_loans,avg_pd,actual_default_rate,avg_actual_rate,avg_annualized_el_rate,avg_required_rate,avg_pricing_gap,underpriced_share,total_expected_profit,avg_rar,total_economic_profit,avg_economic_return
2,AAA,3415,0.0541,0.0419,0.0871,0.0054,0.1154,-0.0283,0.7587,"2,540,546.4292",0.0654,"2,298,015.5597",0.0595
1,AA,8488,0.0871,0.0778,0.1070,0.0092,0.1192,-0.0122,0.5422,"11,983,910.1097",0.1151,"10,847,993.2511",0.1047
0,A,11518,0.1239,0.1235,0.1223,0.0142,0.1242,-0.0018,0.4239,"22,791,989.3793",0.1504,"20,211,759.4955",0.1334
5,BBB,11905,0.1687,0.1661,0.1338,0.0214,0.1314,0.0024,0.3787,"29,903,584.4809",0.1748,"25,324,632.5209",0.1465
4,BB,7727,0.2158,0.2170,0.1455,0.0291,0.1391,0.0063,0.3484,"25,707,692.6279",0.2074,"20,752,990.8452",0.1632
3,B,6987,0.2690,0.2779,0.1570,0.0374,0.1474,0.0097,0.3446,"30,962,017.2079",0.2456,"23,451,355.7222",0.1782
6,CCC,5333,0.3445,0.3531,0.1721,0.0462,0.1562,0.0158,0.2929,"32,778,140.3071",0.3053,"22,397,339.6225",0.1976
7,D,4032,0.4718,0.4826,0.1907,0.0607,0.1707,0.0200,0.2378,"27,501,136.5575",0.3465,"13,795,570.2895",0.1624


In [58]:
# ============================================================
# Policy-based approval strategy comparison using economic profit
# ============================================================

policy_rules = {
    "Conservative": {
        "max_pd": 0.20,
        "excluded_grades": ["CCC", "D"]
    },
    "Balanced": {
        "max_pd": 0.30,
        "excluded_grades": ["D"]
    },
    "Growth": {
        "max_pd": 0.40,
        "excluded_grades": []
    },
    "Aggressive": {
        "max_pd": 0.50,
        "excluded_grades": []
    }
}

policy_rows = []

for policy_name, rule in policy_rules.items():
    approved = test_pricing[
        (test_pricing["calibrated_pd"] <= rule["max_pd"])
        & (~test_pricing["internal_grade"].isin(rule["excluded_grades"]))
    ].copy()

    rejected = test_pricing[~test_pricing.index.isin(approved.index)].copy()

    approved_ead = approved["ead"].sum()

    policy_rows.append({
        "policy": policy_name,
        "max_pd": rule["max_pd"],
        "excluded_grades": ", ".join(rule["excluded_grades"]) if rule["excluded_grades"] else "None",
        "approved_loans": len(approved),
        "rejected_loans": len(rejected),
        "approval_rate": len(approved) / len(test_pricing),
        "approved_ead": approved_ead,
        "avg_pd": approved["calibrated_pd"].mean(),
        "actual_default_rate": approved["default_flag"].mean(),
        "avg_actual_rate": approved["actual_rate"].mean(),
        "avg_required_rate": approved["required_rate"].mean(),

        "total_expected_loss": approved["expected_loss"].sum(),
        "total_expected_profit": approved["expected_profit"].sum(),
        "portfolio_rar": approved["expected_profit"].sum() / approved_ead if approved_ead > 0 else np.nan,

        "total_capital_charge": approved["capital_charge"].sum(),
        "total_collection_cost": approved["collection_cost"].sum(),
        "total_tail_risk_penalty": approved["tail_risk_penalty"].sum(),
        "total_economic_profit": approved["economic_profit"].sum(),
        "portfolio_economic_return": approved["economic_profit"].sum() / approved_ead if approved_ead > 0 else np.nan,

        "underpriced_share": (approved["pricing_status"] == "Underpriced").mean(),
        "high_risk_share": approved["internal_grade"].isin(["B", "CCC", "D"]).mean()
    })

policy_comparison = pd.DataFrame(policy_rows)

display(policy_comparison)

,policy,max_pd,excluded_grades,approved_loans,rejected_loans,approval_rate,approved_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,portfolio_rar,total_capital_charge,total_collection_cost,total_tail_risk_penalty,total_economic_profit,portfolio_economic_return,underpriced_share,high_risk_share
0,Conservative,0.2000,"CCC, D",36383,23022,0.6125,"472,913,075.0000",0.1256,0.1218,0.1198,0.1249,"22,171,888.7400","70,401,426.0250",0.1489,"5,786,167.3434","2,217,188.8740","1,114,804.4314","61,283,265.3763",0.1296,0.4659,0.0000
1,Balanced,0.3000,D,49727,9678,0.8371,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141",0.1830,"12,993,471.6989","4,489,081.9341","3,095,361.2700","101,518,224.4112",0.1522,0.4344,0.1342
2,Growth,0.4000,None,55545,3860,0.9350,"768,067,625.0000",0.1767,0.1760,0.1328,0.1326,"64,016,744.6039","157,799,454.6211",0.2054,"20,002,920.9399","6,401,674.4604","5,421,466.3574","125,973,392.8634",0.1640,0.4194,0.2249
3,Aggressive,0.5000,None,58289,1116,0.9812,"818,221,850.0000",0.1892,0.1888,0.1354,0.1343,"77,362,390.3878","176,585,978.4372",0.2158,"25,182,439.8346","7,736,239.0388","7,501,260.0702","136,166,039.4935",0.1664,0.4110,0.2614


In [59]:
selected_policy_name = "Balanced"
selected_policy = policy_rules[selected_policy_name]

print("Selected dashboard policy:", selected_policy_name)
print(selected_policy)

test_pricing["strategy_approved"] = (
    (test_pricing["calibrated_pd"] <= selected_policy["max_pd"])
    & (~test_pricing["internal_grade"].isin(selected_policy["excluded_grades"]))
)

strategy_dist = (
    test_pricing["strategy_approved"]
    .value_counts()
    .rename(index={True: "Approved", False: "Rejected"})
    .to_frame("count")
    .assign(pct=lambda x: x["count"] / x["count"].sum() * 100)
)

display(strategy_dist)

approved_portfolio = test_pricing[test_pricing["strategy_approved"]].copy()
rejected_portfolio = test_pricing[~test_pricing["strategy_approved"]].copy()

approved_kpi = portfolio_kpi_summary(approved_portfolio, name="approved_balanced_policy")
rejected_kpi = portfolio_kpi_summary(rejected_portfolio, name="rejected_balanced_policy")

display(pd.concat([approved_kpi, rejected_kpi], ignore_index=True))

Selected dashboard policy: Balanced
{'max_pd': 0.3, 'excluded_grades': ['D']}


,count,pct
strategy_approved,,
Approved,49727,83.7084
Rejected,9678,16.2916


,portfolio,n_loans,total_ead,avg_pd,avg_lgd,avg_actual_rate,avg_required_rate,avg_lifetime_el_rate,avg_annualized_el_rate,total_interest_income,total_expected_loss,total_funding_cost,total_operating_cost,total_expected_profit,portfolio_rar,total_capital_charge,total_collection_cost,total_tail_risk_penalty,total_economic_profit,portfolio_economic_return,actual_default_rate,underpriced_share,fairly_priced_share,overpriced_share
0,approved_balanced_policy,49727,"667,123,675.0000",0.1571,0.3891,0.1282,0.1299,0.0651,0.0199,"303,426,690.1550","44,890,819.3409","90,959,821.0000","45,479,910.5000","122,096,139.3141",0.1830,"12,993,471.6989","4,489,081.9341","3,095,361.2700","101,518,224.4112",0.1522,0.1553,0.4344,0.2120,0.3536
1,rejected_balanced_policy,9678,"171,491,625.0000",0.3961,0.5692,0.1798,0.1620,0.2272,0.0520,"149,626,510.3825","39,235,175.0971","32,212,305.0000","16,106,152.5000","62,072,877.7854",0.3620,"14,864,678.3155","3,923,517.5097","5,723,249.0649","37,561,432.8953",0.2190,0.4059,0.2690,0.1715,0.5595


In [60]:
scenario_assumptions = {
    "Base": {
        "funding_cost_rate": 0.04,
        "operating_cost_rate": 0.02,
        "target_margin_rate": 0.05
    },
    "Higher Funding Cost": {
        "funding_cost_rate": 0.06,
        "operating_cost_rate": 0.02,
        "target_margin_rate": 0.05
    },
    "High Margin Target": {
        "funding_cost_rate": 0.04,
        "operating_cost_rate": 0.02,
        "target_margin_rate": 0.08
    },
    "Stress Cost Scenario": {
        "funding_cost_rate": 0.06,
        "operating_cost_rate": 0.03,
        "target_margin_rate": 0.08
    }
}

scenario_policy_rows = []

for scenario_name, scenario_params in scenario_assumptions.items():
    scenario_df = add_pricing_profitability_metrics(
        test_df,
        LGD_BY_GRADE,
        funding_cost_rate=scenario_params["funding_cost_rate"],
        operating_cost_rate=scenario_params["operating_cost_rate"],
        target_margin_rate=scenario_params["target_margin_rate"],
        pricing_tolerance=ASSUMPTIONS["pricing_tolerance"],
        max_allowed_rate=ASSUMPTIONS["max_allowed_rate"],
        min_expected_profit=ASSUMPTIONS["min_expected_profit"],
        capital_cost_rate=ASSUMPTIONS["capital_cost_rate"],
        collection_cost_rate=ASSUMPTIONS["collection_cost_rate"],
        tail_risk_multiplier=ASSUMPTIONS["tail_risk_multiplier"]
    )

    for policy_name, rule in policy_rules.items():
        approved = scenario_df[
            (scenario_df["calibrated_pd"] <= rule["max_pd"])
            & (~scenario_df["internal_grade"].isin(rule["excluded_grades"]))
        ].copy()

        approved_ead = approved["ead"].sum()

        scenario_policy_rows.append({
            "scenario": scenario_name,
            "policy": policy_name,
            "funding_cost_rate": scenario_params["funding_cost_rate"],
            "operating_cost_rate": scenario_params["operating_cost_rate"],
            "target_margin_rate": scenario_params["target_margin_rate"],

            "approval_rate": len(approved) / len(scenario_df),
            "approved_loans": len(approved),
            "approved_ead": approved_ead,
            "avg_pd": approved["calibrated_pd"].mean(),
            "actual_default_rate": approved["default_flag"].mean(),
            "avg_actual_rate": approved["actual_rate"].mean(),
            "avg_required_rate": approved["required_rate"].mean(),

            "total_expected_loss": approved["expected_loss"].sum(),
            "total_expected_profit": approved["expected_profit"].sum(),
            "portfolio_rar": approved["expected_profit"].sum() / approved_ead if approved_ead > 0 else np.nan,

            "total_capital_charge": approved["capital_charge"].sum(),
            "total_collection_cost": approved["collection_cost"].sum(),
            "total_tail_risk_penalty": approved["tail_risk_penalty"].sum(),
            "total_economic_profit": approved["economic_profit"].sum(),
            "portfolio_economic_return": approved["economic_profit"].sum() / approved_ead if approved_ead > 0 else np.nan,

            "total_repriced_economic_profit": approved["repriced_economic_profit"].sum(),
            "repriced_economic_return": approved["repriced_economic_profit"].sum() / approved_ead if approved_ead > 0 else np.nan,

            "underpriced_share": (approved["pricing_status"] == "Underpriced").mean(),
            "high_risk_share": approved["internal_grade"].isin(["B", "CCC", "D"]).mean()
        })

scenario_policy_summary = pd.DataFrame(scenario_policy_rows)

display(scenario_policy_summary)

,scenario,policy,funding_cost_rate,operating_cost_rate,target_margin_rate,approval_rate,approved_loans,approved_ead,avg_pd,actual_default_rate,avg_actual_rate,avg_required_rate,total_expected_loss,total_expected_profit,portfolio_rar,total_capital_charge,total_collection_cost,total_tail_risk_penalty,total_economic_profit,portfolio_economic_return,total_repriced_economic_profit,repriced_economic_return,underpriced_share,high_risk_share
0,Base,Conservative,0.0400,0.0200,0.0500,0.6125,36383,"472,913,075.0000",0.1256,0.1218,0.1198,0.1249,"22,171,888.7400","70,401,426.0250",0.1489,"5,786,167.3434","2,217,188.8740","1,114,804.4314","61,283,265.3763",0.1296,"66,404,435.6012",0.1404,0.4659,0.0000
1,Base,Balanced,0.0400,0.0200,0.0500,0.8371,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1299,"44,890,819.3409","122,096,139.3141",0.1830,"12,993,471.6989","4,489,081.9341","3,095,361.2700","101,518,224.4112",0.1522,"93,121,861.3470",0.1396,0.4344,0.1342
2,Base,Growth,0.0400,0.0200,0.0500,0.9350,55545,"768,067,625.0000",0.1767,0.1760,0.1328,0.1326,"64,016,744.6039","157,799,454.6211",0.2054,"20,002,920.9399","6,401,674.4604","5,421,466.3574","125,973,392.8634",0.1640,"104,937,379.4923",0.1366,0.4194,0.2249
3,Base,Aggressive,0.0400,0.0200,0.0500,0.9812,58289,"818,221,850.0000",0.1892,0.1888,0.1354,0.1343,"77,362,390.3878","176,585,978.4372",0.2158,"25,182,439.8346","7,736,239.0388","7,501,260.0702","136,166,039.4935",0.1664,"108,502,951.0563",0.1326,0.4110,0.2614
4,Higher Funding Cost,Conservative,0.0600,0.0200,0.0500,0.6125,36383,"472,913,075.0000",0.1256,0.1218,0.1198,0.1449,"22,171,888.7400","40,192,387.5250",0.0850,"5,786,167.3434","2,217,188.8740","1,114,804.4314","31,074,226.8763",0.0657,"66,404,435.6012",0.1404,0.6822,0.0000
5,Higher Funding Cost,Balanced,0.0600,0.0200,0.0500,0.8371,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1499,"44,890,819.3409","76,616,228.8141",0.1148,"12,993,471.6989","4,489,081.9341","3,095,361.2700","56,038,313.9112",0.0840,"93,121,861.3470",0.1396,0.6464,0.1342
6,Higher Funding Cost,Growth,0.0600,0.0200,0.0500,0.9350,55545,"768,067,625.0000",0.1767,0.1760,0.1328,0.1526,"64,016,744.6039","103,094,078.1211",0.1342,"20,002,920.9399","6,401,674.4604","5,421,466.3574","71,268,016.3634",0.0928,"104,937,379.4923",0.1366,0.6273,0.2249
7,Higher Funding Cost,Aggressive,0.0600,0.0200,0.0500,0.9812,58289,"818,221,850.0000",0.1892,0.1888,0.1354,0.1543,"77,362,390.3878","117,016,822.4372",0.1430,"25,182,439.8346","7,736,239.0388","7,501,260.0702","76,596,883.4935",0.0936,"108,502,951.0563",0.1326,0.6169,0.2614
8,High Margin Target,Conservative,0.0400,0.0200,0.0800,0.6125,36383,"472,913,075.0000",0.1256,0.1218,0.1198,0.1549,"22,171,888.7400","70,401,426.0250",0.1489,"5,786,167.3434","2,217,188.8740","1,114,804.4314","61,283,265.3763",0.1296,"111,717,993.3512",0.2362,0.7698,0.0000
9,High Margin Target,Balanced,0.0400,0.0200,0.0800,0.8371,49727,"667,123,675.0000",0.1571,0.1553,0.1282,0.1599,"44,890,819.3409","122,096,139.3141",0.1830,"12,993,471.6989","4,489,081.9341","3,095,361.2700","101,518,224.4112",0.1522,"161,341,727.0970",0.2418,0.7353,0.1342


# 7. Save Pricing Datasets and Artifacts

In [61]:
train_pricing_path = PROCESSED_DIR / "train_pricing.csv"
valid_pricing_path = PROCESSED_DIR / "valid_pricing.csv"
test_pricing_path = PROCESSED_DIR / "test_pricing.csv"

train_pricing.to_csv(train_pricing_path, index=False)
valid_pricing.to_csv(valid_pricing_path, index=False)
test_pricing.to_csv(test_pricing_path, index=False)

test_kpi.to_csv(DATA_ARTIFACT_DIR / "portfolio_kpi_summary.csv", index=False)
test_grade_profitability.to_csv(DATA_ARTIFACT_DIR / "grade_profitability_summary.csv", index=False)
test_frontier.to_csv(DATA_ARTIFACT_DIR / "portfolio_frontier.csv", index=False)
scenario_summary.to_csv(DATA_ARTIFACT_DIR / "scenario_summary.csv", index=False)
lgd_table.to_csv(DATA_ARTIFACT_DIR / "lgd_assumptions.csv", index=False)
policy_comparison.to_csv(DATA_ARTIFACT_DIR / "policy_comparison.csv", index=False)
scenario_policy_summary.to_csv(DATA_ARTIFACT_DIR / "scenario_policy_summary.csv", index=False)

print("Saved pricing datasets:")
for path in [train_pricing_path, valid_pricing_path, test_pricing_path]:
    print(f"{path} | {path.stat().st_size / (1024 ** 2):,.2f} MB")

print("\nSaved artifacts to:", DATA_ARTIFACT_DIR)

Saved pricing datasets:
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\train_pricing.csv | 172.76 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\valid_pricing.csv | 37.03 MB
c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\data\processed\test_pricing.csv | 37.32 MB

Saved artifacts to: c:\Users\Anwar\Documents\Work duty\Risk-Based Loan Pricing & Portfolio Profitability Simulator\artifacts\data


In [62]:
# Readable methodology report
methodology_notes = pd.DataFrame([
    {
        "section": "PD",
        "note": "Final PD is produced by raw XGBoost probability because sigmoid calibration worsened Brier Score and ECE."
    },
    {
        "section": "EAD",
        "note": "EAD is proxied by loan_amnt."
    },
    {
        "section": "LGD",
        "note": "LGD is assumption-based by internal grade because recovery data is unavailable."
    },
    {
        "section": "Expected Loss",
        "note": "Expected Loss = calibrated_pd × LGD × EAD."
    },
    {
        "section": "Required Rate",
        "note": "Required Rate = funding cost + operating cost + expected loss rate + target margin."
    },
    {
        "section": "Expected Profit",
        "note": "Expected Profit = interest income - funding cost - operating cost - expected loss."
    },
    {
        "section": "Pricing Gap",
        "note": "Pricing Gap = actual interest rate - required risk-based interest rate."
    },
    {
        "section": "Portfolio Frontier",
        "note": "Portfolio frontier simulates PD approval thresholds and compares approval rate, expected loss, and expected profit."
    }
])

methodology_notes.to_csv(REPORTS_DIR / "05_pricing_methodology_notes.csv", index=False)

display(methodology_notes)

,section,note
0,PD,Final PD is produced by raw XGBoost probabilit...
1,EAD,EAD is proxied by loan_amnt.
2,LGD,LGD is assumption-based by internal grade beca...
3,Expected Loss,Expected Loss = calibrated_pd × LGD × EAD.
4,Required Rate,Required Rate = funding cost + operating cost ...
5,Expected Profit,Expected Profit = interest income - funding co...
6,Pricing Gap,Pricing Gap = actual interest rate - required ...
7,Portfolio Frontier,Portfolio frontier simulates PD approval thres...
